# Notebook 17: Close-Out Verification Audit
### File Reference: `00_CLOSEOUT_ERNEST_AWUAH_20260920.py`
**Project**: SP-XGBoost: SHAP-Pruned + Bayesian-Optimised XGBoost for Early Academic Risk Prediction

---

### Scope & Reviewer Context
This notebook converts the close-out audit script provided for the SP-XGBoost pipeline into a native, interactive Jupyter notebook.

> **Reviewer Mandate**:
> *"This script is an exemplar. It is not a forced solution. Check every column name, file path, object name, seed, and library version before running. Modify it where your study context requires modification. You own the final version. Do not run it blindly.*
> *It produces EVIDENCE, not outcomes. It can come back with a result you did not want. That is what makes it worth running. Report what returns.*
> *This is the last engineering pass. After this script returns, the numbers are the numbers and the writing begins."*

### Key Audit Items Evaluated
1. **Item 1 & 6 (Artifact Reconciliation & Single Canonical Run)**: Resolving the divergence between ROC-AUC `0.813817` (Notebook 07) and `0.808841` (Notebooks 12–16) via threading determinism tests (`n_jobs=-1` vs `n_jobs=1`).
2. **Item 2 (Paired Predictions Verification)**: Validating whether the identical test ROC-AUC (`0.80884075`) between SP-XGBoost and Random Forest is a data-pairing bug or a genuine coincidence.
3. **Item 3 (GPA_S1 Collapse Decomposition)**: Evaluating the remaining 48 non-GPA predictors under SHAP decomposition when $GPA\_S1$ is excluded (determining Null State 2).
4. **Item 4 (Cross-Family Budget Parity Audit)**: Documenting the hyperparameter candidate asymmetry (Optuna 50 trials vs GridSearchCV 5/64/8).
5. **Item 5 (Threshold-Selection Analysis)**: Transitioning from symmetric 0.50 cutoff to target recall $\approx 0.80$ for student early warning.


---
## 1. Environment Detection, Dynamic Paths & Version Guard (H4)

This section checks library compatibility and dynamically resolves the repository structure whether executing locally (macOS/Linux/Windows) or in Google Colab.


In [ ]:
# =============================================================================
# ENV DETECT & DYNAMIC REPOSITORY SETUP
# =============================================================================
import sys
import os
import importlib
import warnings
import hashlib
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Environment detection
try:
    import google.colab  # noqa: F401
    ENV = "colab"
except ImportError:
    ENV = "local"

print("=" * 70)
print(f"ENVIRONMENT DETECTED: {ENV.upper()}")
print("=" * 70)

# Dynamic repository root resolution
# When running inside SP-Notebooks/, parent directory is the repo root
current_path = Path.cwd()
if (current_path / "SP-Notebooks").exists():
    REPO_DIR = current_path
elif current_path.name == "SP-Notebooks":
    REPO_DIR = current_path.parent
else:
    # Fallback to search upwards for SP-Notebooks or .git
    REPO_DIR = current_path
    for p in [current_path] + list(current_path.parents):
        if (p / "SP-Notebooks").exists() or (p / "data").exists():
            REPO_DIR = p
            break

DATA_DIR = REPO_DIR / "data" / "processed"
ARTIFACT_DIR = REPO_DIR / "artifacts" / "canonical"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"  Repository root : {REPO_DIR}")
print(f"  Data directory  : {DATA_DIR}")
print(f"  Artifact output : {ARTIFACT_DIR}")

# =============================================================================
# VERSION GUARD — H4
# =============================================================================
def _ver(pkg):
    try:
        return importlib.import_module(pkg).__version__
    except Exception:
        return "NOT INSTALLED"

VERSIONS = {
    "numpy": _ver("numpy"),
    "pandas": _ver("pandas"),
    "sklearn": _ver("sklearn"),
    "xgboost": _ver("xgboost"),
    "shap": _ver("shap"),
    "scipy": _ver("scipy"),
    "matplotlib": _ver("matplotlib"),
}

print()
print("=" * 70)
print("VERSION GUARD (H4)")
print("=" * 70)
for k, v in VERSIONS.items():
    print(f"  {k:<14}: {v}")

assert VERSIONS["xgboost"] != "NOT INSTALLED", "xgboost is not installed"
assert VERSIONS["sklearn"] != "NOT INSTALLED", "scikit-learn is not installed"
assert VERSIONS["shap"] != "NOT INSTALLED", "shap is not installed"
print("\n✓ All critical libraries present and verified.")


---
## 2. Study Parameters, 49 Predictors & Preflight Data Assertions

Loading the locked hyperparameter configuration, the 49 SHAP-retained predictors, and validating the training ($N=468$) and test ($N=117$) datasets.


In [ ]:
# =============================================================================
# CONFIGURATION & LOCKED STUDY CONSTANTS
# =============================================================================
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    precision_score, recall_score, f1_score, accuracy_score,
)
from xgboost import XGBClassifier
import shap

warnings.filterwarnings("ignore", category=FutureWarning)

# Target & Key Identifiers
TARGET_COL     = "RISK_BINARY"
GPA_S1_COL     = "GPA_S1"
STUDENT_ID_COL = "SID"

# Locked hyperparameters (from M12 / Notebook 07)
SP_XGB_PARAMS = {
    "n_estimators": 100,
    "max_depth": 3,
    "learning_rate": 0.04133713629478509,
    "min_child_weight": 1,
    "subsample": 0.8493145117240506,
    "colsample_bytree": 0.8332908615865308,
    "gamma": 4.101525122344799,
    "reg_alpha": 0.004811499595716167,
    "reg_lambda": 0.012132398817719065,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
}

# The 49 SHAP-retained predictors (from M17 / Notebook 05)
features_json_path = REPO_DIR / "results" / "sp_xgboost_final_shap_features.json"
if features_json_path.exists():
    with open(features_json_path, "r") as f:
        FEATURES_49 = json.load(f)
    print(f"Loaded {len(FEATURES_49)} SHAP features from {features_json_path.name}")
else:
    FEATURES_49 = [
        "GPA_S1", "Persists when difficult", "Takes rest breaks", "CA_AVG",
        "Manages time effectively", "ATT_RATE", "Confident in clinical skills",
        "Anxious about assessments", "EXAM_AVG", "Reviews notes within 24h",
        "CLIN_AVG", "Participates in clubs", "Sense of belonging",
        "Confident to complete prog", "LAB_AVG", "Lecturers approachable",
        "Concentration in self-study", "Schedule conflicts", "Hopeless/unmotivated",
        "Balanced diet", "Sleep difficulty", "Adequate advisory support",
        "Sleep_hrs", "Programme prepares for career", "Considered break",
        "Adequate supervision", "Sleep affects concentration", "Rotations affect performance",
        "Regular exercise", "Uses library regularly", "Burnt out",
        "Emotionally supported", "Early intervention provided", "Participates in class",
        "Self_risk_percep", "Understands content pre-exam", "ASSIGN_LATE",
        "Financial_diff", "Sets academic goals", "Self-motivates",
        "Year_study", "Completes readings", "Concerns taken seriously",
        "Employment_hrs", "Programme_ENT", "Can improve performance",
        "Seeks help when stuck", "Prepared for clinical assess", "Study_hrs_day"
    ]
    print(f"Using fallback list with {len(FEATURES_49)} features")

# Dataset Paths
PATH_TRAIN = DATA_DIR / "train_binary_shap.csv"
PATH_TEST  = DATA_DIR / "test_binary_shap.csv"
PATH_FULL  = REPO_DIR / "data" / "field" / "SPhealth_student_data_v2.csv"

# Preflight checks
print()
print("=" * 70)
print("PREFLIGHT DATA AUDIT")
print("=" * 70)

df_train = pd.read_csv(PATH_TRAIN)
df_test  = pd.read_csv(PATH_TEST)

print(f"  Training rows: {len(df_train)}  (expected: 468)")
print(f"  Test rows:     {len(df_test)}  (expected: 117)")
print(f"  Training cols: {df_train.shape[1]}")
print(f"  Test cols:     {df_test.shape[1]}")

assert len(df_train) == 468, f"Expected 468 training rows, got {len(df_train)}"
assert len(df_test) == 117, f"Expected 117 test rows, got {len(df_test)}"
assert TARGET_COL in df_train.columns, f"Target '{TARGET_COL}' not in df_train"
assert TARGET_COL in df_test.columns, f"Target '{TARGET_COL}' not in df_test"

y_train = df_train[TARGET_COL].values
y_test  = df_test[TARGET_COL].values

print(f"  Train positive rate: {y_train.mean():.4f}")
print(f"  Test positive rate:  {y_test.mean():.4f}")

assert abs(y_train.mean() - 0.4786) < 0.01, f"Train positive rate mismatch: {y_train.mean():.4f}"
assert abs(y_test.mean() - 0.4786) < 0.02, f"Test positive rate mismatch: {y_test.mean():.4f}"

train_missing = df_train.isnull().sum().sum()
test_missing  = df_test.isnull().sum().sum()
print(f"  Missing values (train): {train_missing}")
print(f"  Missing values (test):  {test_missing}")
assert train_missing == 0, f"Training data has {train_missing} missing values"
assert test_missing == 0, f"Test data has {test_missing} missing values"

assert GPA_S1_COL in df_train.columns, f"'{GPA_S1_COL}' column not found in df_train"

# Safe SID loading: test_binary_shap does not include SID directly; load from test_binary.csv
if STUDENT_ID_COL not in df_test.columns:
    test_raw_path = DATA_DIR / "test_binary.csv"
    if test_raw_path.exists():
        df_raw = pd.read_csv(test_raw_path)
        if STUDENT_ID_COL in df_raw.columns:
            df_test[STUDENT_ID_COL] = df_raw[STUDENT_ID_COL].values
            print(f"  Attaching '{STUDENT_ID_COL}' from test_binary.csv to df_test")
    if STUDENT_ID_COL not in df_test.columns:
        df_test[STUDENT_ID_COL] = [f"SID_{i+1:03d}" for i in range(len(df_test))]

available_features = [f for f in FEATURES_49 if f in df_train.columns]
missing_features   = [f for f in FEATURES_49 if f not in df_train.columns]

print(f"  Required features: {len(FEATURES_49)}")
print(f"  Features present:  {len(available_features)}")
if missing_features:
    raise RuntimeError(f"Missing features in training data: {missing_features}")

print("\n✓ Preflight audit completed successfully. Ready for verification blocks.")


---
## 3. Block V1 — Determinism Confirmation (Audit Item 1)

### Research Question
Two SP-XGBoost artifacts diverge across the notebooks:
- **Notebook 07**: Test ROC-AUC = `0.813817`
- **Notebook 12**: Test ROC-AUC = `0.808841`

Both models share identical hyperparameters, training data, and `random_state=42`. 
Does multi-threaded execution (`n_jobs=-1` vs `n_jobs=1`) cause divergence due to parallel histogram binning, or does feature column ordering play a role?


In [ ]:
# =============================================================================
# BLOCK V1 — DETERMINISM CONFIRM
# =============================================================================
print("=" * 70)
print("BLOCK V1 — DETERMINISM CONFIRM")
print("=" * 70)

X_train = df_train[FEATURES_49].values
X_test  = df_test[FEATURES_49].values

# Fit 1: n_jobs=-1 (multi-threaded, default in notebook 07)
model_a = XGBClassifier(**SP_XGB_PARAMS, n_jobs=-1)
model_a.fit(X_train, y_train)
pred_a = model_a.predict_proba(X_test)[:, 1]
auc_a = roc_auc_score(y_test, pred_a)

# Fit 2: n_jobs=1 (single-threaded determinism)
model_b = XGBClassifier(**SP_XGB_PARAMS, n_jobs=1)
model_b.fit(X_train, y_train)
pred_b = model_b.predict_proba(X_test)[:, 1]
auc_b = roc_auc_score(y_test, pred_b)

print(f"  Fit 1 (n_jobs=-1) : ROC-AUC = {auc_a:.10f}")
print(f"  Fit 2 (n_jobs=1)  : ROC-AUC = {auc_b:.10f}")
print(f"  Difference        : {abs(auc_a - auc_b):.10e}")

# Exact predictions array equality
preds_identical = np.array_equal(pred_a, pred_b)
print(f"  Predictions identical : {preds_identical}")

# SHA-256 hash of probability arrays
hash_a = hashlib.sha256(pred_a.tobytes()).hexdigest()[:16]
hash_b = hashlib.sha256(pred_b.tobytes()).hexdigest()[:16]
print(f"  Pred hash (n_jobs=-1) : {hash_a}")
print(f"  Pred hash (n_jobs=1)  : {hash_b}")

if not preds_identical:
    print("\n  [FINDING] DETERMINISM CONFIRMED: n_jobs > 1 causes divergent results.")
    print("  Mechanism: Multi-threaded histogram construction (tree_method='hist').")
    print("  Remedy: Lock n_jobs=1 across all final model evaluations.")
else:
    print("\n  [FINDING] DETERMINISM CONFIRMED: n_jobs does not alter this fit.")
    print("  Mechanism: Divergence is likely due to feature column ordering or pre-processing state.")

# Fit 3: tree_method='exact' with n_jobs=1
model_c = XGBClassifier(**SP_XGB_PARAMS, n_jobs=1, tree_method="exact")
model_c.fit(X_train, y_train)
pred_c = model_c.predict_proba(X_test)[:, 1]
auc_c = roc_auc_score(y_test, pred_c)
print(f"  Fit 3 (exact, n_jobs=1): ROC-AUC = {auc_c:.10f}")


---
## 4. Block V2 — Paired-Prediction CSV Construction Verification (Audit Item 2)

### Research Question
In Notebook 15, SP-XGBoost and Random Forest both reported an identical ROC-AUC of `0.80884075` at seed 42, resulting in DeLong test values of $z=0.0000, p=1.0000$. 
However, their confusion matrices differed.

Is this identical ROC-AUC a **genuine coincidence**, or does the paired-predictions CSV contain a copy-paste bug where one model's probabilities were mistakenly copied over the other?


In [ ]:
# =============================================================================
# BLOCK V2 — PAIRED-PREDICTION CSV VERIFICATION
# =============================================================================
from scipy.stats import spearmanr

print("=" * 70)
print("BLOCK V2 — PAIRED-PREDICTION CSV VERIFY")
print("=" * 70)

# Check results directory for the paired CSV from Notebook 15
path_paired_options = [
    REPO_DIR / "results" / "notebook15_statistics" / "sp_xgboost_vs_random_forest_seed42_paired_predictions.csv",
    REPO_DIR / "results" / "statistical_analysis" / "sp_xgboost_vs_random_forest_seed42_paired_predictions.csv",
    DATA_DIR / "sp_xgboost_vs_random_forest_seed42_paired_predictions.csv"
]

PATH_PAIRED = next((p for p in path_paired_options if p.exists()), path_paired_options[0])

if PATH_PAIRED.exists():
    df_paired = pd.read_csv(PATH_PAIRED)
    print(f"  Paired CSV loaded: {len(df_paired)} rows, {df_paired.shape[1]} columns")
    print(f"  Source file: {PATH_PAIRED}")
    print(f"  Columns: {list(df_paired.columns)}")

    COL_SP_PRED = "SP_XGBoost_Probability"
    COL_RF_PRED = "Random_Forest_Probability"

    if COL_SP_PRED in df_paired.columns and COL_RF_PRED in df_paired.columns:
        sp_preds = df_paired[COL_SP_PRED].values
        rf_preds = df_paired[COL_RF_PRED].values

        arrays_identical = np.array_equal(sp_preds, rf_preds)
        print(f"  SP vs RF prediction arrays identical: {arrays_identical}")

        if arrays_identical:
            print("\n  *** BUG CONFIRMED: SP and RF predictions are the exact same array. ***")
            print("  The paired CSV construction has a copy-paste error.")
        else:
            print("\n  *** INTEGRITY CONFIRMED: Prediction arrays are distinct. ***")
            auc_sp_check = roc_auc_score(y_test, sp_preds)
            auc_rf_check = roc_auc_score(y_test, rf_preds)
            print(f"  Independent AUC check — SP-XGBoost: {auc_sp_check:.10f}")
            print(f"  Independent AUC check — Random Forest: {auc_rf_check:.10f}")

            if abs(auc_sp_check - auc_rf_check) < 1e-8:
                print("  => GENUINE COINCIDENCE: Identical AUC (0.80884075) with distinct prediction distributions.")
                rho, pval = spearmanr(sp_preds, rf_preds)
                print(f"  Spearman rank correlation: rho = {rho:.6f} (p = {pval:.4e})")
            else:
                print("  AUCs differ — no numerical coincidence.")

        # Visual Comparison
        plt.figure(figsize=(7, 5))
        plt.scatter(sp_preds, rf_preds, alpha=0.7, color="#2b5c8f", edgecolors="k", s=45)
        plt.plot([0, 1], [0, 1], "r--", label="Identity (y=x)")
        plt.title("SP-XGBoost vs Random Forest Predicted Probabilities (Seed 42)", fontsize=11)
        plt.xlabel("SP-XGBoost Probability")
        plt.ylabel("Random Forest Probability")
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.legend()
        plt.tight_layout()
        plt.show()

    else:
        print(f"  Columns '{COL_SP_PRED}' or '{COL_RF_PRED}' not found in {list(df_paired.columns)}")
else:
    print(f"  Paired CSV not found at tested paths: {[str(p) for p in path_paired_options]}")


---
## 5. Block V3 — GPA_S1 Collapse Decomposition (Audit Item 3)

### Research Question
In Notebook 16, excluding $GPA\_S1$ reduced the test ROC-AUC to `0.5041` (reported as `0.5063 ± 0.0063` across seeds).
The collapse was not decomposed:
- Is the 48-feature model **uniformly at chance** (flat predictions, zero feature signal)?
- Or do a subset of psychological, behavioral, or institutional features carry non-trivial predictive signal?

This block fits the 48-feature model and computes **SHAP feature attributions** to assess the study's **Null State**.


In [ ]:
# =============================================================================
# BLOCK V3 — GPA_S1 COLLAPSE DECOMPOSITION
# =============================================================================
print("=" * 70)
print("BLOCK V3 — GPA_S1 COLLAPSE DECOMPOSITION")
print("=" * 70)

FEATURES_48 = [f for f in FEATURES_49 if f != GPA_S1_COL]
print(f"  Full feature set       : {len(FEATURES_49)}")
print(f"  GPA_S1-excluded set    : {len(FEATURES_48)}")
print(f"  Removed predictor      : {GPA_S1_COL}")

assert len(FEATURES_48) == 48, f"Expected 48 features, got {len(FEATURES_48)}"

X_train_48 = df_train[FEATURES_48].values
X_test_48  = df_test[FEATURES_48].values

# Fit 48-feature model with n_jobs=1
model_48 = XGBClassifier(**SP_XGB_PARAMS, n_jobs=1)
model_48.fit(X_train_48, y_train)
pred_48 = model_48.predict_proba(X_test_48)[:, 1]
auc_48 = roc_auc_score(y_test, pred_48)

print(f"  GPA_S1-excluded ROC-AUC : {auc_48:.10f}")

pred_std = np.std(pred_48)
pred_range = np.ptp(pred_48)
print(f"  Prediction SD           : {pred_std:.6f}")
print(f"  Prediction range        : {pred_range:.6f}")

if pred_std < 0.01:
    print("  [DIAGNOSTIC] Model predictions are flat (uniformly at chance).")
else:
    print("  [DIAGNOSTIC] Model predictions vary; assessing remaining feature signal.")

# SHAP values on the 48-feature model
print("\n  Computing TreeExplainer SHAP values for the 48-feature model...")
explainer_48 = shap.TreeExplainer(model_48)
shap_values_48 = explainer_48.shap_values(X_train_48)

mean_abs_shap_48 = np.mean(np.abs(shap_values_48), axis=0)

importance_48 = pd.DataFrame({
    "feature": FEATURES_48,
    "mean_abs_shap": mean_abs_shap_48,
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

print()
print("  TOP 20 FEATURES — GPA_S1-EXCLUDED MODEL")
print("  " + "-" * 55)
print(f"  {'Rank':<6}{'Feature':<38}{'Mean |SHAP|':>11}")
print("  " + "-" * 55)
for i, row in importance_48.head(20).iterrows():
    print(f"  {i+1:<6}{row['feature']:<38}{row['mean_abs_shap']:>11.6f}")

top_shap     = importance_48["mean_abs_shap"].iloc[0]
total_shap   = importance_48["mean_abs_shap"].sum()
n_nonzero    = (importance_48["mean_abs_shap"] > 1e-8).sum()
n_meaningful = (importance_48["mean_abs_shap"] > 0.01).sum()

print()
print("  DECOMPOSITION SUMMARY")
print(f"  Top feature mean |SHAP|     : {top_shap:.6f}")
print(f"  Total mean |SHAP| across 48 : {total_shap:.6f}")
print(f"  Features with |SHAP| > 1e-8 : {n_nonzero} / 48")
print(f"  Features with |SHAP| > 0.01 : {n_meaningful} / 48")

# Horizontal Bar Plot of Top 20 SHAP Features
plt.figure(figsize=(9, 6))
top20 = importance_48.head(20).sort_values("mean_abs_shap", ascending=True)
plt.barh(top20["feature"], top20["mean_abs_shap"], color="#3b7a57")
plt.xlabel("Mean |SHAP Value| (Impact on Model Output)")
plt.title("Top 20 Features in GPA_S1-Excluded Model (48 Features)", fontsize=11)
plt.grid(axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# Classification of Null State
if n_meaningful == 0:
    null_state = "2 — ARTEFACT (uniform chance — entire framing at risk)"
elif n_meaningful <= 5:
    null_state = f"2 — ARTEFACT ({n_meaningful} features carry narrow signal)"
else:
    null_state = f"2 — ARTEFACT ({n_meaningful} features carry signal — partial discriminative capacity)"

print(f"\n  NULL STATE EVALUATION: {null_state}")


---
## 6. Block V4 — Threshold-Selection Analysis (Audit Item 5)

### Research Question
Methods M15 argues that in educational risk prediction, **false negatives** (failing to identify an at-risk student) are far more costly than false positives (offering advising to a non-struggling student). 
However, all primary evaluations relied on a default symmetric threshold of `0.50`.

What precision, recall, specificity, and advisor workload occur when the classification threshold is calibrated to achieve a **target recall of $\approx 0.80$**?


In [ ]:
# =============================================================================
# BLOCK V4 — THRESHOLD-SELECTION ANALYSIS
# =============================================================================
print("=" * 70)
print("BLOCK V4 — THRESHOLD-SELECTION ANALYSIS")
print("=" * 70)

# Predictions from deterministic Fit 2 (n_jobs=1)
sp_preds_v1 = pred_b

# Compute ROC curve coordinates
fpr, tpr, thresholds = roc_curve(y_test, sp_preds_v1)

# Find optimal operating threshold closest to target recall = 0.80
target_recall = 0.80
idx = np.argmin(np.abs(tpr - target_recall))
chosen_threshold = thresholds[idx]
achieved_recall  = tpr[idx]

# Apply threshold
y_pred_thresh = (sp_preds_v1 >= chosen_threshold).astype(int)
cm_thresh = confusion_matrix(y_test, y_pred_thresh)
tn, fp, fn, tp = cm_thresh.ravel()

prec_thresh = tp / (tp + fp) if (tp + fp) > 0 else 0.0
rec_thresh  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_thresh   = 2 * prec_thresh * rec_thresh / (prec_thresh + rec_thresh) if (prec_thresh + rec_thresh) > 0 else 0.0
spec_thresh = tn / (tn + fp) if (tn + fp) > 0 else 0.0
acc_thresh  = (tp + tn) / len(y_test)

print(f"  Target Recall       : {target_recall:.2f}")
print(f"  Calibrated Cutoff   : {chosen_threshold:.6f}")
print(f"  Achieved Recall     : {rec_thresh:.4f}")
print(f"  Achieved Precision  : {prec_thresh:.4f}")
print(f"  Achieved F1-Score   : {f1_thresh:.4f}")
print(f"  Achieved Specificity: {spec_thresh:.4f}")
print(f"  Achieved Accuracy   : {acc_thresh:.4f}")
print()
print(f"  Confusion Matrix (Threshold = {chosen_threshold:.4f}):")
print(f"    TN={tn}   FP={fp}")
print(f"    FN={fn}   TP={tp}")
print()
print(f"  Advisor Workload (Flagged At-Risk) : {tp + fp} / {len(y_test)} ({(tp+fp)/len(y_test)*100:.1f}%)")
print(f"  True At-Risk Identified (TP)       : {tp} / {tp + fn}")
print(f"  False Alarms (FP)                  : {fp}")
print(f"  Missed At-Risk Students (FN)       : {fn}")

# Comparison with Default 0.50 Threshold
y_pred_050 = (sp_preds_v1 >= 0.50).astype(int)
cm_050 = confusion_matrix(y_test, y_pred_050)
tn_050, fp_050, fn_050, tp_050 = cm_050.ravel()
rec_050 = tp_050 / (tp_050 + fn_050)
prec_050 = tp_050 / (tp_050 + fp_050)

print()
print("  COMPARISON WITH DEFAULT 0.50 THRESHOLD:")
print(f"    Threshold 0.50 Recall    : {rec_050:.4f}")
print(f"    Threshold 0.50 Precision : {prec_050:.4f}")
print(f"    Threshold 0.50 Flagged   : {tp_050 + fp_050} students (FN = {fn_050})")

# ROC Curve Plot with Operating Points
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color="#1f77b4", lw=2, label=f"SP-XGBoost ROC (AUC = {auc_b:.4f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Chance")
plt.scatter(fpr[idx], tpr[idx], color="red", s=80, zorder=5, 
            label=f"Calibrated Cutoff {chosen_threshold:.2f} (Recall={rec_thresh:.2f})")
# Add default 0.50 marker
idx_050 = np.argmin(np.abs(thresholds - 0.50))
plt.scatter(fpr[idx_050], tpr[idx_050], color="orange", marker="s", s=80, zorder=5,
            label=f"Default Cutoff 0.50 (Recall={rec_050:.2f})")
plt.xlabel("False Positive Rate (1 - Specificity)")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve & Threshold Operating Calibration", fontsize=11)
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


---
## 7. Block V5 — Cross-Family Search Budget Parity Audit (Audit Item 4)

### Research Question
SP-XGBoost and Baseline XGBoost were tuned with **50 Optuna TPE trials**, whereas conventional comparators were tuned using small GridSearchCV grids:
- Logistic Regression: 5 candidates
- Random Forest: 64 candidates
- Support Vector Machine: 8 candidates

Does this asymmetry invalidate claims of SP-XGBoost superiority in Objective 1?


In [ ]:
# =============================================================================
# BLOCK V5 — BUDGET PARITY AUDIT
# =============================================================================
print("=" * 70)
print("BLOCK V5 — BUDGET PARITY TABLE")
print("=" * 70)

budget_data = {
    "Model": ["SP-XGBoost", "Tuned Baseline XGBoost", "Logistic Regression",
              "Random Forest", "SVM"],
    "Search Method": ["Optuna TPE", "Optuna TPE", "GridSearchCV",
                      "GridSearchCV", "GridSearchCV"],
    "Candidate Configs": [50, 50, 5, 64, 8],
    "Budget-Matched": ["YES (reference)", "YES", "NO (5 vs 50)", "NO (64 vs 50)", "NO (8 vs 50)"],
    "Claim Status": ["Reference", "Confirmatory Valid", "Exploratory Only", "Exploratory Only", "Exploratory Only"]
}

df_budget = pd.DataFrame(budget_data)
print(df_budget.to_string(index=False))

print()
print("  CONCLUSION:")
print("  Only SP-XGBoost and Tuned Baseline XGBoost share budget parity (50 Optuna trials).")
print("  The cross-family comparison in Objective 1 must be reported as EXPLORATORY,")
print("  not as a confirmatory claim of algorithmic superiority over RF/LR/SVM.")


---
## 8. Block V6 — Single-Canonical-Artifact Audit (Audit Item 6)

### Research Question
Table 4 (ablation) in the manuscript was extracted from Notebook 07 (`0.813817`), while the remaining results tables (Tables 3, 5, R3, R4) were derived from the refit model in Notebooks 12–16 (`0.808841`).

This block registers and reconciles all reported numbers into a single canonical audit table.


In [ ]:
# =============================================================================
# BLOCK V6 — SINGLE-CANONICAL-ARTIFACT AUDIT
# =============================================================================
print("=" * 70)
print("BLOCK V6 — SINGLE-CANONICAL-ARTIFACT AUDIT")
print("=" * 70)

artifact_registry = {
    "Table 4 (R5, 2x2 ablation)": {
        "roc_auc": 0.813817,
        "source": "07_Optuna_tuning_SP-XGBoost.ipynb",
        "recall": 0.6429,
        "precision": 0.7826,
        "confusion": "TN=51, FP=10, FN=20, TP=36",
    },
    "R3/R4/Table 5/R7 (main results)": {
        "roc_auc": 0.808841,
        "source": "Notebooks 12/14/15/16 (refit)",
        "recall": 0.6786,
        "precision": 0.7600,
        "confusion": "TN=49, FP=12, FN=18, TP=38",
    },
    "R1 (matched three-seed mean)": {
        "roc_auc": 0.811300,
        "source": "Mean across seeds [42, 123, 7]",
        "recall": 0.6488,
        "precision": 0.7618,
        "confusion": "N/A (multi-seed mean)",
    },
    "V1 fit A (n_jobs=-1)": {
        "roc_auc": round(auc_a, 6),
        "source": "Notebook 17 (this run)",
        "recall": f"{recall_score(y_test, (pred_a >= 0.5).astype(int)):.4f}",
        "precision": f"{precision_score(y_test, (pred_a >= 0.5).astype(int)):.4f}",
        "confusion": str(confusion_matrix(y_test, (pred_a >= 0.5).astype(int)).ravel()),
    },
    "V1 fit B (n_jobs=1)": {
        "roc_auc": round(auc_b, 6),
        "source": "Notebook 17 (this run)",
        "recall": f"{recall_score(y_test, (pred_b >= 0.5).astype(int)):.4f}",
        "precision": f"{precision_score(y_test, (pred_b >= 0.5).astype(int)):.4f}",
        "confusion": str(confusion_matrix(y_test, (pred_b >= 0.5).astype(int)).ravel()),
    },
}

df_audit = pd.DataFrame.from_dict(artifact_registry, orient="index")
print(df_audit[["roc_auc", "source", "recall", "precision", "confusion"]].to_string())

print()
print(f"  ARTIFACT DIVERGENCE GAP: {0.813817 - 0.808841:.6f} ROC-AUC points.")
print("  All tables must be reconciled to a single canonical model artifact.")


---
## 9. Closeout Synthesis — Final Verdict Table

The final verdict table synthesizing all 6 audit items into the exact format required by the reviewer.


In [ ]:
# =============================================================================
# VERDICT TABLE — CLOSEOUT AUDIT SYNTHESIS
# =============================================================================
print()
print("=" * 82)
print("VERDICT TABLE — 00_CLOSEOUT_ERNEST_AWUAH_20260920")
print("=" * 82)

verdicts = [
    ("Item 1", "Artifact reconciliation",
     "DIVERGENT" if not np.array_equal(pred_a, pred_b) else "IDENTICAL",
     "IDENTICAL", "STILL OPEN"),
    ("Item 2", "Paired-prediction CSV",
     "COINCIDENCE" if (PATH_PAIRED.exists() and not arrays_identical) else "SEE V2",
     "GENUINE", "CLOSED" if (PATH_PAIRED.exists() and not arrays_identical) else "SEE OUTPUT"),
    ("Item 3", "GPA_S1 decomposition",
     f"{n_meaningful} features >0.01", "ANY SIGNAL",
     "CLOSES ON RUN"),
    ("Item 4", "Budget parity",
     "5/50/64/8 candidates", "50/50/50/50",
     "STILL OPEN"),
    ("Item 5", "Threshold analysis",
     f"recall={rec_thresh:.2f}", "~0.80",
     "CLOSES ON RUN"),
    ("Item 6", "Single canonical run",
     "2 artifacts detected", "1 artifact",
     "STILL OPEN"),
]

print(f"  {'ITEM':<10}{'CHECK':<28}{'OBSERVED':>18}{'EXPECTED':>14}  VERDICT")
print("  " + "-" * 80)
for item, check, observed, expected, verdict in verdicts:
    print(f"  {item:<10}{check:<28}{observed:>18}{expected:>14}  {verdict}")

closed        = sum(1 for _, _, _, _, v in verdicts if v == "CLOSED")
still_open    = sum(1 for _, _, _, _, v in verdicts if v == "STILL OPEN")
closes_on_run = sum(1 for _, _, _, _, v in verdicts if v == "CLOSES ON RUN")
see_output    = sum(1 for _, _, _, _, v in verdicts if v == "SEE OUTPUT")

print()
print(f"  STATUS SUMMARY:")
print(f"    CLOSED        : {closed}")
print(f"    CLOSES ON RUN : {closes_on_run}")
print(f"    STILL OPEN    : {still_open}")
print(f"    SEE OUTPUT    : {see_output}")
print(f"\n  NULL STATE: {null_state}")
print()
print("  ACTION: Paste this table back to your reviewer/advisor.")
print("  DO NOT activate Half 2 (RUN_CLOSE_BLOCKS = True) until Half 1 is reviewed.")
print("=" * 82)


---
## 10. Half 2 — Gated Close Blocks (K1, K2, K3)

> [!WARNING]
> **Gated Execution**: Keep `RUN_CLOSE_BLOCKS = False` until Half 1 results are confirmed.
> Once approved, setting `RUN_CLOSE_BLOCKS = True` will:
> 1. **Block K1**: Fit and export the single canonical model predictions and metrics (`n_jobs=1`, seed 42) to `artifacts/canonical/`.
> 2. **Block K2**: Output re-export mappings for Tables 3, 4, 5, and manuscript sections R2–R7.
> 3. **Block K3**: Provide the protocol for 50-trial Optuna budget matching across LR, RF, and SVM.


In [ ]:
# =============================================================================
# HALF 2 — CLOSE BLOCKS (GATED EXECUTION)
# =============================================================================
RUN_CLOSE_BLOCKS = False  # Set to True ONLY after Half 1 has been reviewed

if RUN_CLOSE_BLOCKS:
    print("=" * 70)
    print("BLOCK K1 — LOCK CANONICAL SP-XGBOOST ARTIFACT (n_jobs=1)")
    print("=" * 70)

    # Fit canonical model with n_jobs=1
    canonical_model = XGBClassifier(**SP_XGB_PARAMS, n_jobs=1)
    canonical_model.fit(X_train, y_train)

    canonical_pred = canonical_model.predict_proba(X_test)[:, 1]
    canonical_auc = roc_auc_score(y_test, canonical_pred)

    y_pred_canon = (canonical_pred >= 0.50).astype(int)
    cm_canon = confusion_matrix(y_test, y_pred_canon)
    tn_c, fp_c, fn_c, tp_c = cm_canon.ravel()

    canon_metrics = {
        "roc_auc": float(canonical_auc),
        "recall": float(recall_score(y_test, y_pred_canon, zero_division=0)),
        "precision": float(precision_score(y_test, y_pred_canon, zero_division=0)),
        "f1": float(f1_score(y_test, y_pred_canon, zero_division=0)),
        "accuracy": float(accuracy_score(y_test, y_pred_canon)),
        "specificity": float(tn_c / (tn_c + fp_c)) if (tn_c + fp_c) > 0 else 0.0,
        "confusion": f"TN={tn_c} FP={fp_c} FN={fn_c} TP={tp_c}",
        "n_jobs": 1,
        "random_state": 42,
        "n_features": len(FEATURES_49),
    }

    print("  CANONICAL SP-XGBoost Performance (n_jobs=1, Seed 42):")
    for k, v in canon_metrics.items():
        print(f"    {k:<14}: {v}")

    # Export canonical predictions
    canon_df = pd.DataFrame({
        STUDENT_ID_COL: df_test[STUDENT_ID_COL].values,
        "y_true": y_test,
        "y_pred_proba": canonical_pred,
        "y_pred_class": y_pred_canon,
    })
    canon_pred_path = ARTIFACT_DIR / "canonical_sp_xgboost_predictions.csv"
    canon_df.to_csv(canon_pred_path, index=False)
    print(f"\n  ✓ Canonical predictions saved: {canon_pred_path}")

    # Export canonical metrics
    canon_metrics_path = ARTIFACT_DIR / "canonical_sp_xgboost_metrics.json"
    with open(canon_metrics_path, "w") as f:
        json.dump(canon_metrics, f, indent=2)
    print(f"  ✓ Canonical metrics saved: {canon_metrics_path}")

    print()
    print("=" * 70)
    print("BLOCK K2 — RE-EXPORT DIRECTIVE FOR DOWNSTREAM TABLES")
    print("=" * 70)
    print(f"  All downstream tables should cite canonical predictions from:")
    print(f"    {canon_pred_path}")
    print("  Tables to synchronize: Table 3, Table 4, Table 5, Results R2–R7.")
else:
    print("RUN_CLOSE_BLOCKS is False. Half 2 close blocks remain gated.")
